<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 07 · 把工作步骤变成可导出的 Skill

团队已经整理好 CSV 金额校验步骤。先直接创建 Skill，并把第一版导出成可以打开的 SKILL.md。随后用审核流程加入一项新步骤，比较服务版本与已导出的文件，最后标记这份 Skill 不再推荐使用。

**完成后你能做到：** 直接创建 managed Skill，核对标准包和导出文件，审核后续版本，并区分内容 revision、导出副本和生命周期状态。

预计 20 分钟。先按 [README](README.md) 安装环境；本篇可以独立运行，不依赖其他 Notebook 的变量或数据。本篇无需模型和 API Key。

按顺序读说明、运行代码，再对照结果。练习可以改输入；完整重跑使用 **Restart Kernel & Run All**。

## 准备本篇实验

这格启动一个回环地址的真实 Server，并创建独立 Scope，默认把数据保存在本篇自己的 SQLite 文件中，也可按 [README](README.md#使用-oceanbase-运行) 显式选择专用 OceanBase 测试库。
`_tutorial.py` 只管理环境和显示结果；下面的业务调用都是可在应用中复用的公开 API。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))


if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("07", features=())
client = lab.client
assert client is not None

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 07",
        summary="第 07 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-07",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 把整理好的步骤直接创建为 Skill

name、description、instructions 描述如何找到和使用这份操作说明；validation 说明如何判断执行结果。
通过 `create_artifact` 提交后，服务保存第一版 Skill，并生成对应的标准包。
本例内容已经由我们明确整理，不需要先创建 Candidate。

In [ ]:
from powercontext.http import ArtifactReference, CreateArtifactRequest, SkillProposal

proposal = SkillProposal.model_validate({
    "name": "csv-amount-validation",
    "description": "在实现或评审订单 CSV 的 amount 字段时，组织金额精度与边界检查。",
    "instructions": (
        "# CSV 金额校验\n\n"
        "1. 读取当前项目约定，确认金额单位、允许范围和精度。\n"
        "2. 整理正常、空白、非数字、负数、超精度输入及预期结果。\n"
        "3. 使用项目公开入口执行检查，保留输入类别和实际结果。\n"
        "4. 检查坏行提示是否包含原始行号，避免回显整行敏感内容。\n"
        "5. 报告已检查和未检查的范围；未执行的检查不能标记为通过。"
    ),
    "validation": ["正常金额按约定单位转换", "非法金额明确拒绝", "检查结果可以追溯"],
})

created = await client.create_artifact(
    scope_id,
    CreateArtifactRequest.model_validate({
        "family": "skill",
        "content": proposal.model_dump(mode="json"),
    }),
)
skill_ref = ArtifactReference(family="skill", artifact_id=created.artifact_id, revision=created.revision)
skill = await client.get_artifact(scope_id, "skill", created.artifact_id)
assert skill is not None and skill.content["package"] is not None
show({"名称": skill.content["name"], "版本": skill.revision, "已形成标准包": True})

## 2. 已有 Skill，是否就会进入模型的上下文？

这个 Scope 只有 Skill。`prepare_context` 应返回 empty，因为当前 PreparedContext 从 Memory 和 Experience 选择材料。
Skill 通过自己的包与使用流程交给 Host；创建成功不意味着已安装或执行。

In [ ]:
from powercontext.http import PrepareContextRequest

context = await client.prepare_context(PrepareContextRequest(scope_id=scope_id, query="amount", max_bytes=5000))
assert context.status == "empty"
print("Skill 已保存；本 Scope 尚无可供 PreparedContext 召回的 Memory 或 Experience。")

## 3. 导出一个精确版本，打开实际文件

下面调用公开 CLI，将第一版导出到本篇实验目录。Python 子进程对应普通终端里的 `powercontext skill export`。
导出目标是课堂目录，不是正在使用的 Agent 安装目录。

打开实际 `SKILL.md` 后，再用公开包清单中的 SHA-256 核对文件，确认它来自刚才指定的版本。

In [ ]:
import asyncio
import hashlib
import os

from powercontext.http import GetSkillPackageRequest

destination = lab.directory / "exported-skills" / proposal.name
environment = dict(os.environ)
environment["POWERCONTEXT_CLIENT_SERVER_URL"] = lab.base_url
environment.pop("POWERCONTEXT_CLIENT_API_TOKEN", None)
command = [
    sys.executable,
    "-c",
    "from powercontext.cli.app import main; main()",
    "skill",
    "export",
    skill_ref.artifact_id,
    "--scope-id",
    scope_id,
    "--revision",
    str(skill_ref.revision),
    "--target",
    "codex",
    "--destination",
    str(destination),
]
process = await asyncio.create_subprocess_exec(
    *command,
    env=environment,
    stdout=asyncio.subprocess.PIPE,
    stderr=asyncio.subprocess.PIPE,
)
stdout, stderr = await process.communicate()
assert process.returncode == 0, stderr.decode()
assert (destination / "SKILL.md").is_file()
print((destination / "SKILL.md").read_text(encoding="utf-8"))


manifest = await client.get_skill_package_manifest(GetSkillPackageRequest(scope_id=scope_id, artifact=skill_ref))
markdown_file = next(item for item in manifest.files if item.path == "SKILL.md")
assert hashlib.sha256((destination / "SKILL.md").read_bytes()).hexdigest() == markdown_file.digest
show({
    "精确版本": skill_ref.model_dump(mode="json"),
    "包文件数": manifest.package.file_count,
    "导出内容与包清单一致": True,
})

## 4. 加一项新步骤，先给审核者看

现在希望补上空文件检查。把新规范保存成 Source，为已有 Skill 提交 replacement Candidate。
`source_refs` 引用新规范，`artifact_refs` 引用原 Skill；`target` 指向要更新的精确版本。
直接写入自动生成的系统 Source 只记录该版本的来源，不能当作新候选的工作证据。

候选存在期间，读取当前 Skill 仍应得到第一版。先检查提案，再运行下一格批准。

In [ ]:
from powercontext.http import CreateSourceRequest, ProposeSkillRequest, SourceReference

followup = await client.create_source(
    scope_id, CreateSourceRequest(content="CSV 校验规范补充：空文件应给出明确提示，并加入用例表。")
)
replacement = await client.propose_skill(
    ProposeSkillRequest(
        scope_id=scope_id,
        target=skill_ref,
        proposal=proposal.model_copy(
            update={"instructions": proposal.instructions + "\n6. 增加空文件用例，确认提示清楚。"}
        ),
        source_refs=[
            SourceReference(name="content", source_id=followup.source_id),
        ],
        artifact_refs=[skill_ref],
        reason="加入空文件检查规范",
    )
)
before = await client.get_artifact(scope_id, "skill", created.artifact_id)
assert replacement.status == "pending" and before is not None and before.revision == 1
show(replacement.proposal)

## 5. 批准后，哪一份内容变了？

本格批准的是上面刚展示的教学提案。批准后，当前制品变为新版本；指定 revision=1 仍可读取旧版。
先前导出的文件也应保持原样。要让 Host 使用新步骤，需要选择新版本再次导出或使用对应分发流程。

In [ ]:
from powercontext.http import ApproveArtifactCandidateRequest

successor = await client.approve_artifact_candidate(
    ApproveArtifactCandidateRequest(
        scope_id=scope_id,
        candidate_id=replacement.candidate_id,
        expected_version=replacement.version,
    )
)
assert successor.result_artifact is not None
newer = await client.get_artifact(scope_id, "skill", created.artifact_id)
older = await client.get_artifact_revision(scope_id, "skill", created.artifact_id, 1)
assert newer is not None and newer.revision > older.revision
assert "空文件" not in older.content["instructions"] and "空文件" in newer.content["instructions"]
assert "空文件" not in (destination / "SKILL.md").read_text(encoding="utf-8")
table([
    {"位置": "Server 第一版", "Revision": older.revision, "含空文件步骤": False},
    {"位置": "Server 当前版", "Revision": newer.revision, "含空文件步骤": True},
    {"位置": "已导出的文件", "Revision": skill_ref.revision, "含空文件步骤": False},
])

## 练习：标记不再推荐的 Skill

设想项目已经改用另一套校验流程。将这份 Skill 标记为 deprecated，然后显式列出包含 deprecated 的库。
观察它的内容 revision 与治理状态：生命周期变化不等于改写 SKILL.md，也不会删除已导出的文件。

In [ ]:
from powercontext.http import ListManagedSkillsRequest, UpdateSkillLifecycleRequest

library = await client.list_managed_skills(ListManagedSkillsRequest(scope_id=scope_id))
current_skill = next(item for item in library.skills if item.artifact.artifact_id == skill_ref.artifact_id)
governance = await client.update_skill_lifecycle(
    UpdateSkillLifecycleRequest(
        scope_id=scope_id,
        artifact_id=skill_ref.artifact_id,
        expected_generation=current_skill.governance.governance_generation,
        lifecycle_state="deprecated",
    )
)
assert governance.lifecycle_state == "deprecated"
visible = await client.list_managed_skills(ListManagedSkillsRequest(scope_id=scope_id, include_deprecated=True))
table([
    {"名称": item.content.name, "内容版本": item.artifact.revision, "状态": item.governance.lifecycle_state}
    for item in visible.skills
])
retained = await client.get_artifact_revision(scope_id, "skill", created.artifact_id, 1)
assert retained.content == older.content
assert (destination / "SKILL.md").is_file()

## 保存收获，关闭连接

Skill 可以直接创建；候选审核用于待判断的变更。制品版本、标准包、导出文件和治理状态都有各自可观察的结果。

下面关闭本篇 Client 和 Server，保留实验文件供检查。中途停止时也可运行这一格；清理方式见 [README](README.md#清理实验数据)。

下一篇：[08](08_automatic_extraction.ipynb)。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")